# Russell’s Scout · Project walkthrough

Scout searches WorkBC for data and ML jobs, classifies the results, and uses employer job-board adapters to enrich them. PostgreSQL keeps the production history; the daily digest turns that history into a readable report. A separate Claude Code agent can work on adapter repairs. The regular scouting pipeline does not call an LLM.

**Start here:** Run sections 0–6 in order. They follow the source with small, editable examples and checked-in fixtures. Section 0 downloads the public repo, Python and dependencies; after that, the core demos use no live sources. No GPU, Docker, PostgreSQL server, or API key is needed. Sections 7–8 are optional and disabled by default. Section 9 is a reading lab for the Ubuntu deployment, not a Colab setup.

| Path | What moves through it |
| --- | --- |
| `sources/workbc.py` → `pipeline/record.py` → `sweep.py` | WorkBC JSON → classified records → sweep summary |
| `sources/ats.py` → `enrich.py` → `trust.py` | Employer boards → normalized details → scored evidence |
| `store.py` and `migrations/` → `digest.py` | PostgreSQL history → digest |
| `crawl.py` and `extract.py` | HTML/JavaScript → CSS extraction → postings |
| `agent/session.sh` and `agent/workspace-step.sh` | Source-health signals → gated repair work → reviewable branch bundle |

The Python paths in this table are relative to `src/scout/`. The notebook checks out a fixed revision so changes to `main` do not silently change the examples.

## 0 · Set up the source checkout

The project requires Python 3.12 or newer. Use a separate Python 3.12 environment so Scout's pins do not replace libraries in the notebook kernel. Later cells call Scout in that environment with `in_repo` and `run_python`; they do not import it into the notebook kernel. The core path uses the same browser-free requirements as Scout’s production runtime plus the repository’s test tools.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path
from textwrap import dedent

REV = "c7c6b85bd3716010041cce2e0ddc2be41832ed62"
URL = "https://github.com/xqzr2000/russells-scout.git"
root = Path("/content") if Path("/content").is_dir() else Path.cwd()
repo = root / f"scout-walkthrough-{REV[:8]}"
venv = root / f"scout-venv-{REV[:8]}"

def run(*args, **kwargs):
    return subprocess.run([str(arg) for arg in args], check=True, **kwargs)

if not repo.exists():
    run("git", "init", "-q", repo)
head = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=repo, capture_output=True, text=True
)
if head.returncode != 0 or head.stdout.strip() != REV:
    run("git", "fetch", "--depth=1", URL, REV, cwd=repo)
    run("git", "checkout", "--detach", "FETCH_HEAD", cwd=repo)

uv = shutil.which("uv")
if uv is None:
    run(sys.executable, "-m", "pip", "install", "uv")
    uv = shutil.which("uv") or str(Path(sys.executable).with_name("uv"))

python = venv / "bin" / "python"
if not python.exists():
    run(uv, "venv", "--python", "3.12", venv)

def install(*args):
    run(uv, "pip", "install", "--python", python, *args, cwd=repo)

install("-r", "requirements-runtime.txt", "-r", "requirements-dev.txt")
install("--no-deps", "-e", ".")

# Keep examples pointed at the checked-in search settings, not a host config.
env = {key: value for key, value in os.environ.items() if not key.startswith("SCOUT_")}
env.pop("PYTHONPATH", None)
env["SCOUT_CONFIG"] = str(repo / "config/search.json")

def in_repo(*args):
    return run(python, *args, cwd=repo, env=env)

def run_python(source):
    return in_repo("-c", dedent(source).strip())

in_repo("--version")
run("git", "rev-parse", "--short", "HEAD", cwd=repo)
in_repo("-m", "scout.cli", "--help")

## 1 · Follow one WorkBC page into Scout records

`pipeline/record.py:build` uses `config/search.json` for title and area matching, then classifies the outbound link. A job can have `area="remote"` without the source explicitly labeling the workplace remote; these are separate fields. Compare the three fixture records before looking at the whole sweep.

In [ ]:
run_python("""
    import json
    from pathlib import Path
    from scout import config
    from scout.pipeline.record import build

    page = json.loads(Path("tests/fixtures/workbc/page_mixed.json").read_text())
    cfg = config.load()
    rows = [build(job, ["data scientist"], cfg) for job in page["result"]]
    for row in rows:
        print(f"{row.title} | {row.area}/{row.workplace} | {row.match_quality} | "
              f"{row.ats}:{row.ats_board or '-'} | relevant={row.relevant}")
        print("  remote signals:", row.remote_signs or "(none)")
    assert len(rows) == 3 and all(row.relevant for row in rows)
""")

**Try it:** Print `rows[0].matched_terms` and `rows[0].exact_terms`. The former says which query brought in the posting; the latter says which configured title terms actually matched.

## 2 · Replay a sweep without hitting WorkBC

A saved sweep names raw pages by search term and page number. Give the real CLI one fixture in that layout. `--replay` replaces the HTTP transport with a file reader; `--dry-run` prevents database writes. The summary should report three unique postings: one Metro Vancouver and two remote.

In [ ]:
from tempfile import TemporaryDirectory

with TemporaryDirectory(prefix="scout-replay-") as temp:
    folder = Path(temp)
    shutil.copyfile(
        repo / "tests/fixtures/workbc/page_mixed.json",
        folder / "data_scientist-p1.json",
    )
    in_repo("-m", "scout.cli", "sweep", "--dry-run", "--no-save",
            "--term", "data scientist", "--replay", folder)

**Try it:** Rename the saved page in the cell, rerun, and read the missing-page error. A replay should not fill a gap by contacting WorkBC. Restore the filename before continuing.

## 3 · Read an employer board and reproduce adapter failures

ATS means applicant tracking system. Use the real Ashby adapter with a fake HTTP getter that serves one checked-in response. It should leave out the unlisted analyst role. Then run the repository’s Workday and Amazon regressions for malformed rows: a usable row can survive a bad neighbor; an all-unusable page is a structural failure, not a genuinely empty board.

In [ ]:
run_python("""
    from pathlib import Path
    from scout.sources import ats

    url = ats.ASHBY.format(board="northwind")
    data = Path("tests/fixtures/ats/ashby_northwind.json").read_bytes()

    def fixture_get(request_url):
        assert request_url == url, request_url
        return data

    board = ats.fetch_board("ashby", "northwind", get=fixture_get, sleep=lambda _: None)
    assert board.outcome == "ok", board.error
    for posting in board.postings:
        print(f"{posting.posting_id} | {posting.title} | {posting.workplace} | {posting.location}")
    assert "Data Analyst (internal)" not in {p.title for p in board.postings}
""")
in_repo("-m", "pytest", "-q", "tests/test_ats.py",
        "-k", "unusable_item or no_usable_item")

## 4 · Change one input and inspect the trust-score evidence

The trust score is a heuristic, not a verification of a posting or employer. A negative score flags a posting for review rather than deleting it. Hold the job and its ATS link fixed; change only the description.

In [ ]:
run_python("""
    import json
    from dataclasses import replace
    from pathlib import Path
    from scout import config, sweep, trust
    from scout.pipeline.record import build

    raw = json.loads(Path("tests/fixtures/workbc/page_mixed.json").read_text())["result"][0]
    original = sweep.trust_input(build(raw, ["data scientist"], config.load()))
    for description in ("Build forecasting models with Python and SQL.",
                        "To apply, pay an upfront fee."):
        score, reasons = trust.score(replace(original, description=description))
        print(f"{description} -> {score:+d}: {', '.join(reasons)}")
""")

**Try it:** Replace the fee text with “Experience building Telegram bots” and then “Contact us on Telegram.” Read the reasons rather than inferring the result from the presence of an app name alone.

## 5 · Render a digest without connecting to PostgreSQL

Use the real digest renderer with a synthetic digest object. In production the database and health logs provide the inputs. Here there is only one fixture-style job and no database connection, file write, or notification.

In [ ]:
run_python("""
    from datetime import UTC, datetime, timedelta
    from scout.digest import Digest, Job, NEW, Section

    now = datetime(2026, 11, 1, 15, tzinfo=UTC)
    report = Digest(
        covers_from=now - timedelta(days=1), covers_to=now,
        day="Sun Nov 1", stamp="2026-11-01-0700", since="Sat Oct 31 07:00",
        first=False, relevant_open=1,
        sections=[Section(NEW, "strong",
                          [Job("Data Scientist", "Example & Co", "Vancouver", None)], 1)],
        health=["sweeps: 2 of 2 ok (example)"],
        problems=["Example: an employer board needs review"],
    )
    print(report.text())
    assert "Example &amp; Co" in report.html()
""")

## 6 · Run the selected offline checks

These cover WorkBC, classification, links, sweep replay, ATS adapters, deduplication, trust and digest rendering. Some digest tests require a database and are intentionally skipped in this environment. This is not the full deployment suite; `tests/test_extract.py` belongs to the optional browser-dependency lab.

In [ ]:
in_repo("-m", "pytest", "-q",
        "tests/test_workbc.py", "tests/test_classify.py", "tests/test_links.py",
        "tests/test_sweep.py", "tests/test_ats.py", "tests/test_dupes.py",
        "tests/test_trust.py", "tests/test_digest.py")

## 7 · Optional: compare plain HTTP with Chromium

The regular WorkBC and ATS sweep consumes JSON and does not need a browser. Crawl4AI is a Python library used by the development/browser path; it is not a separate Scout service in `compose.yaml`. `crawl.py` launches or reuses a crawler and applies the CSS schema from `extract.py`. On the same bundled HTML, the plain HTTP test sees three postings; Chromium runs the JavaScript that adds a fourth, **Data Engineer**. The CLI writes sample JSON to the checkout’s `output/` directory.

Set `RUN_BROWSER_LAB` to `True` to download the pinned browser dependencies and Chromium. If Playwright reports missing system libraries, use the repository’s Dev Container/Codespaces or install the Playwright system dependencies on a supported Linux image. No live site is used here.

In [ ]:
RUN_BROWSER_LAB = False

if RUN_BROWSER_LAB:
    install("-r", "requirements.txt")
    in_repo("-m", "playwright", "install", "chromium")
    in_repo("-m", "pytest", "-q", "tests/test_extract.py")
    in_repo("-m", "scout.cli", "sample")
    import json
    output = max((repo / "output").glob("sample-jobs-*.json"),
                 key=lambda path: path.stat().st_mtime_ns)
    print(json.dumps(json.loads(output.read_text()), indent=2)[:3000])
else:
    print("Skipped: set RUN_BROWSER_LAB = True to try the bundled page.")

## 8 · Optional: send one live WorkBC search

This can make multiple page requests for one configured term. It does not write to Postgres or save raw pages. Results depend on the live endpoint; if it refuses or the format changes, use the offline replay rather than bypassing restrictions.

In [ ]:
RUN_LIVE_WORKBC = False

if RUN_LIVE_WORKBC:
    in_repo("-m", "scout.cli", "sweep", "--dry-run", "--no-save", "--no-enrich",
            "--term", "data scientist")
else:
    print("Skipped: the offline replay in section 2 covers the same classification path.")

## 9 · Ubuntu box reading lab: from a failure to a reviewed repair

Read [the architecture plan](https://github.com/xqzr2000/russells-scout/blob/c7c6b85bd3716010041cce2e0ddc2be41832ed62/docs/architecture-plan.md), [the box setup](https://github.com/xqzr2000/russells-scout/blob/c7c6b85bd3716010041cce2e0ddc2be41832ed62/docs/box-setup.md) and [the Phase 3 plan](https://github.com/xqzr2000/russells-scout/blob/c7c6b85bd3716010041cce2e0ddc2be41832ed62/docs/phase3-plan.md) for the operational steps. Colab is not running the timers or testing the agent’s security boundary.

1. **Find the evidence.** `enrich.py` records source outcomes; `scout status` exposes failing sources. `agent/session.sh` prepares the briefing and can select a repair for a structural failure, repeated failures, or a degraded source. Restricted or disabled sources are excluded. An existing pending repair can suppress duplicate work.
2. **Trace the permissions.** In `compose.yaml` the agent reads the main checkout, locked policy, and briefing through read-only mounts. Its writable workspace is separate from the checked-out production code, and it does not share the database network. The no-LLM after-step is given the outbox mount to package branches; the agent does not directly push or deploy a fix.
3. **Reproduce before changing code.** Identify whether the failure is in the WorkBC client, an ATS JSON adapter, or the separate HTML extraction path. Save a sanitized fixture, assert the expected behavior in the relevant test, and review the proposed branch before promoting it. Section 3 is a small example of the same fixture-first workflow.
4. **Locate model settings.** The `agent` service in `compose.yaml` sets `ANTHROPIC_MODEL` and the default/subagent aliases; the token comes from the box’s `secrets/deepseek.env`. Model settings apply to agent sessions, not the sweep. Don’t copy the token or deployment environment into Colab.

**Checkpoint:** Would a malformed Workday JSON row send you to `crawl.py` or `sources/ats.py`? What evidence distinguishes one unusable row from a source-wide format change?